# Exercise 05 — Attention

Last week's models read a text one token at a time and had to squeeze everything they had seen into **one hidden state**. You saw what that costs: information from the beginning of a sequence has to survive every step in between. **Attention** removes that bottleneck. Every position looks *directly* at every other position and decides for itself how much each one matters.

In this notebook you build the attention mechanism from scratch, one small piece at a time. There is no training and nothing to download, and **every cell runs in a second or two on a CPU**. In the next notebook you use these pieces to build a complete Transformer.

## What you will do
1. **Attention by hand** — implement scaled dot-product attention in four lines and apply it to a four-word sentence.
2. **Why divide by $\sqrt{d_k}$?** — measure what happens to the softmax without the scaling.
3. **Masking** — stop the model from looking at the future, or at padding.
4. **Order does not matter (yet)** — show that self-attention is blind to word order, and fix it with positional encodings.

## How to work through it
- Run the cells **in order** and fill in **every `# TODO`**.
- Tasks are numbered (**1.1**, **1.2**, …). Tasks that say *Your answer here* want a short written answer, not code.
- Most implementation tasks are followed by a **✅ Check** cell that verifies your work automatically. Run it and make sure it passes before you move on.

Please update your environment before running this week's notebooks using <code onclick="navigator.clipboard.writeText(this.textContent)" style="cursor:pointer" title="Click to copy">uv sync</code> — notebook 2 needs a few new libraries.

In [ ]:
%matplotlib inline
import math

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

## 1. Attention by hand

Attention works with three sets of vectors:

- a **query** $q$ — *what am I looking for?*
- one **key** $k_j$ per word in the context — *what do I contain?*
- one **value** $v_j$ per word in the context — *what do I pass on if you pick me?*

The query is compared with every key, the scores are turned into weights that sum to 1, and the output is the weighted average of the values. With all queries stacked in a matrix $Q$ (and likewise $K$ and $V$) this is one line:

\begin{align}
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
\end{align}

**1.1 Complete `scaled_dot_product_attention`.** It is four steps, one line each.

Hints:
- `K.transpose(-2, -1)` swaps the last two dimensions. Using negative indices means the function also works when there are extra batch dimensions in front — you will need that in the next notebook.
- The softmax runs over the **keys**, which is the last dimension of the score matrix.

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    # Q: (..., n_queries, d_k)   K: (..., n_keys, d_k)   V: (..., n_keys, d_v)
    # Returns the output (..., n_queries, d_v) and the attention weights (..., n_queries, n_keys).
    d_k = Q.size(-1)

    scores = ...   # TODO 1. raw scores: one dot product per (query, key) pair -> (..., n_queries, n_keys)
    scores = ...   # TODO 2. scale by the square root of d_k
    weights = ...  # TODO 3. softmax over the keys
    output = ...   # TODO 4. weighted sum of the values

    return output, weights

Let us try it on the sentence ***He sat by the river bank***. To keep things small we leave out *by* and *the*, and every vector has only two dimensions. The word *bank* is ambiguous, so it sends out a query to find out which context it is in.

In [ ]:
words = ["He", "sat", "river", "bank"]

Q = torch.tensor([[1.0, 0.0]])  # the query of "bank"
K = torch.tensor([
    [1.0, 0.0],  # key of "He"
    [0.0, 1.0],  # key of "sat"
    [1.0, 1.0],  # key of "river"
    [1.0, 0.0],  # key of "bank"
])
V = torch.tensor([
    [1.0, 0.0],  # value of "He"
    [0.0, 1.0],  # value of "sat"
    [0.5, 0.5],  # value of "river"
    [1.0, 0.0],  # value of "bank"
])

output, weights = scaled_dot_product_attention(Q, K, V)

print("Attention of 'bank' over the sentence:")
for word, weight in zip(words, weights[0]):
    print(f"  {word:<6} {weight:.3f}")
print("Output:", output)

In [ ]:
# ✅ Check your attention
assert weights.shape == (1, 4) and output.shape == (1, 2), "wrong shapes: expected weights (1, 4) and output (1, 2)"
assert torch.allclose(weights.sum(dim=-1), torch.ones(1)), "the weights of a query must sum to 1 — softmax over the last dimension"
assert torch.allclose(weights, torch.tensor([[0.2863, 0.1412, 0.2863, 0.2863]]), atol=1e-4), "wrong weights — did you scale by sqrt(d_k)?"
assert torch.allclose(output, F.scaled_dot_product_attention(Q, K, V)), "the output differs from PyTorch's own implementation"
print("Looks good ✅")

**1.2 *He*, *river* and *bank* get exactly the same weight, although their keys are not the same.** Why? And why does *sat* get the least?

---

*Your answer here:*

---

### Self-attention

So far only *bank* asked a question. In **self-attention** every word of the sentence sends out a query, and they are all processed at once: $Q$ simply gets one row per word.

**1.3 Let every word attend to every word.** Use the keys also as queries (`Q = K`) and plot the resulting weight matrix.

In [ ]:
def plot_attention(weights, x_labels, y_labels, title="", ax=None):
    # Heatmap of a (n_queries, n_keys) weight matrix: one row per query, one column per key.
    if ax is None:
        _, ax = plt.subplots(figsize=(0.6 * len(x_labels) + 1.5, 0.6 * len(y_labels) + 1))
    ax.imshow(weights.detach(), cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(x_labels)), x_labels)
    ax.set_yticks(range(len(y_labels)), y_labels)
    ax.set_xlabel("key (attended to)")
    ax.set_ylabel("query (attending)")
    ax.set_title(title)


self_output, self_weights = ...  # TODO: attention with the keys K used as queries as well

print(self_weights)
plot_attention(self_weights, words, words, "Self-attention")

In [ ]:
# ✅ Check your self-attention
assert self_weights.shape == (4, 4), "there should be one row of weights per word"
assert torch.allclose(self_weights.sum(dim=-1), torch.ones(4)), "every row must sum to 1"
assert torch.allclose(self_weights[3], weights[0]), "the row of 'bank' should be what you computed in 1.1"
print("Looks good ✅")

**1.4 With `Q = K` the matrix of raw scores $QK^\top$ is symmetric:** *river* scores *sat* exactly as high as *sat* scores *river*. Is the matrix of attention *weights* symmetric as well? Why (not)?

---

*Your answer here:*

---

## 2. Why divide by $\sqrt{d_k}$?

The division by $\sqrt{d_k}$ looks like a detail. To see what it does, look at the size of a dot product $q \cdot k = \sum_{i=1}^{d_k} q_i k_i$ when the vectors get longer. At initialisation the entries of $q$ and $k$ are roughly independent with mean 0 and variance 1.

**2.1 Measure the variance of the dot product for increasing $d_k$, with and without the scaling.**

Hint: for two batches of vectors `q` and `k` of shape `(n, d_k)`, the row-wise dot products are `(q * k).sum(dim=-1)`.

In [ ]:
torch.manual_seed(0)
n = 10_000

variances = {}
for d_k in [4, 16, 64, 256, 1024]:
    q = torch.randn(n, d_k)
    k = torch.randn(n, d_k)
    dots = ...    # TODO: the n dot products q_1·k_1, ..., q_n·k_n, shape (n,)
    scaled = ...  # TODO: the same, divided by the square root of d_k
    variances[d_k] = (dots.var().item(), scaled.var().item())
    print(f"d_k = {d_k:>4}:   variance of q·k = {variances[d_k][0]:>7.1f}     after scaling = {variances[d_k][1]:.2f}")

In [ ]:
# ✅ Check your variances
for d_k, (raw, scaled_var) in variances.items():
    assert 0.9 * d_k < raw < 1.1 * d_k, f"d_k = {d_k}: the variance of the raw dot product should be close to d_k"
    assert 0.9 < scaled_var < 1.1, f"d_k = {d_k}: after scaling the variance should be close to 1"
print("Looks good ✅")

**2.2 Run the cell below.** It draws one query and ten keys with $d_k = 512$ and shows the attention weights with and without the scaling.

In [ ]:
torch.manual_seed(1)
d_k = 512
q = torch.randn(1, d_k)
keys = torch.randn(10, d_k)

scores = q @ keys.T
unscaled_weights = F.softmax(scores, dim=-1)[0]
scaled_weights = F.softmax(scores / math.sqrt(d_k), dim=-1)[0]

fig, axs = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
axs[0].bar(range(10), unscaled_weights)
axs[0].set_title(f"without scaling (largest weight: {unscaled_weights.max():.3f})")
axs[1].bar(range(10), scaled_weights)
axs[1].set_title(f"with scaling (largest weight: {scaled_weights.max():.3f})")
for ax in axs:
    ax.set_xlabel("key")
axs[0].set_ylabel("attention weight")
plt.tight_layout()
plt.show()

**2.3 Explain what you see, using your result from 2.1.** Why is the left picture a problem for *training* — after all, a model that attends to exactly one word does not sound so bad?

---

*Your answer here:*

---

## 3. Masking

Sometimes a query must not see every key:

- A model that generates text from left to right must not look at **future** words during training, otherwise it could simply copy the answer. This is the **causal mask**.
- Sentences in a batch are padded to the same length, and nobody should attend to **`<pad>`** tokens. This is the **padding mask**.

Both work the same way: before the softmax, the scores of all forbidden (query, key) pairs are set to $-\infty$. Because $e^{-\infty} = 0$, these keys end up with a weight of exactly 0, and the remaining weights still sum to 1.

**3.1 Complete `masked_attention`.** It is your function from 1.1 plus one line.

Hint: `mask` is a boolean tensor, `True` where attention is **allowed**. `scores.masked_fill(condition, value)` writes `value` wherever `condition` is `True`, and `~mask` negates a boolean tensor.

In [ ]:
def masked_attention(Q, K, V, mask=None):
    # mask: boolean, broadcastable to (..., n_queries, n_keys); True = the query may attend to the key.
    d_k = Q.size(-1)
    scores = ...  # TODO: the scaled scores, as in 1.1
    if mask is not None:
        scores = ...  # TODO: set the scores to -inf wherever the mask is False
    weights = ...  # TODO: softmax over the keys
    return weights @ V, weights

**3.2 Build a causal mask for our four words and apply it.** Word $i$ may attend to words $0, \ldots, i$ — itself and everything before it.

Hint: `torch.tril` keeps the lower triangle of a matrix (including the diagonal) and sets the rest to zero. Start from `torch.ones(L, L, dtype=torch.bool)`.

In [ ]:
L = len(words)
causal_mask = ...  # TODO: (L, L) boolean matrix, True on and below the diagonal
print(causal_mask)

_, causal_weights = ...  # TODO: self-attention as in 1.3, with the causal mask
plot_attention(causal_weights, words, words, "Causal self-attention")

In [ ]:
# ✅ Check your causal mask
assert causal_mask.dtype == torch.bool and causal_mask.shape == (4, 4), "the mask should be a (4, 4) boolean tensor"
assert causal_mask[2].tolist() == [True, True, True, False], "word 2 may attend to words 0, 1 and 2"
assert torch.all(causal_weights.triu(diagonal=1) == 0), "no weight may fall on future words"
assert torch.allclose(causal_weights.sum(dim=-1), torch.ones(4)), "every row must still sum to 1"
assert causal_weights[0].tolist() == [1.0, 0.0, 0.0, 0.0], "the first word can only attend to itself"
print("Looks good ✅")

**3.3 Now the padding mask.** Below, the sentence has been padded to length 6. Build a mask that stops every query from attending to the two `<pad>` positions.

Hint: the mask has to say, for every **key**, whether it is a real token. A mask of shape `(1, n_keys)` is broadcast over all queries, so you do not need to build the full matrix.

In [ ]:
PAD = 0
padded_words = words + ["<pad>", "<pad>"]
token_ids = torch.tensor([5, 8, 3, 7, PAD, PAD])

torch.manual_seed(0)
X = torch.randn(6, 8)  # some embeddings for the six positions

pad_mask = ...  # TODO: (1, 6) boolean tensor, True where token_ids is a real token
print(pad_mask)

_, pad_weights = masked_attention(X, X, X, mask=pad_mask)
plot_attention(pad_weights, padded_words, padded_words, "Self-attention with a padding mask")

In [ ]:
# ✅ Check your padding mask
assert pad_mask.shape == (1, 6) and pad_mask.dtype == torch.bool, "the mask should be a (1, 6) boolean tensor"
assert torch.all(pad_weights[:, 4:] == 0), "no query may attend to a <pad> key"
assert torch.allclose(pad_weights.sum(dim=-1), torch.ones(6)), "every row must still sum to 1"
print("Looks good ✅")

**3.4 Two questions about the plot.**
1. Why do we set the *scores* to $-\infty$ before the softmax, instead of simply setting the *weights* to 0 after it?
2. The two `<pad>` rows are not masked: as *queries*, the `<pad>` positions happily attend to the sentence. Why is that not a problem?

---

*Your answer here:*

---

## 4. Order does not matter (yet)

An RNN knows the order of the words because it reads them one after the other. Attention looks at all words at once — what does it know about their order?

**4.1 Shuffle the words of a sentence and run self-attention on the original and on the shuffled sentence.** Then compare the two outputs.

Hint: `X[perm]` reorders the rows of `X`. Use the embeddings as queries, keys and values.

In [ ]:
torch.manual_seed(0)
X = torch.randn(5, 8)                # embeddings of a five-word sentence
perm = torch.tensor([3, 0, 4, 1, 2])  # a new word order

out, _ = ...           # TODO: self-attention on X
out_shuffled, _ = ...  # TODO: self-attention on the shuffled sentence X[perm]

print("Same output, in the new order?", torch.allclose(out_shuffled, out[perm], atol=1e-6))

In [ ]:
# ✅ Check
assert out.shape == (5, 8) and out_shuffled.shape == (5, 8), "wrong shapes"
assert torch.allclose(out_shuffled, out[perm], atol=1e-6), "shuffling the input should only shuffle the output"
assert not torch.allclose(out_shuffled, out, atol=1e-6), "out_shuffled should be the attention output for X[perm]"
print("Looks good ✅")

**4.2 What does this result mean for the sentences *dog bites man* and *man bites dog*?** And does adding learned projections $W_Q$, $W_K$, $W_V$, more heads or more layers change anything about it?

---

*Your answer here:*

---

### Positional encodings

The fix is as simple as it gets: **add a vector that depends on the position** to every word embedding. The original Transformer uses fixed sine and cosine waves of different frequencies:

\begin{align}
PE_{(pos,\, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right), \qquad PE_{(pos,\, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)
\end{align}

Even dimensions get a sine, odd dimensions the cosine of the same frequency, and the frequency drops as the dimension index grows.

**4.3 Complete `PositionalEncoding`.**

Hints:
- `position` has shape `(max_len, 1)` and `div_term` has shape `(d_model / 2,)`, so `position * div_term` is a `(max_len, d_model / 2)` matrix with one row per position and one column per frequency. `div_term` is $1 / 10000^{2i/d_{model}}$, computed in a numerically stable way.
- `pe[:, 0::2]` selects the even columns, `pe[:, 1::2]` the odd ones.
- `register_buffer` stores a tensor in the module **without** making it a trainable parameter.
- In `forward`, the input is shorter than `max_len`: add only the first `seq_len` positions.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        position = torch.arange(max_len, dtype=torch.float).unsqueeze(1)                          # (max_len, 1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))  # (d_model / 2,)

        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = ...  # TODO: even dimensions: sine of position * div_term
        pe[:, 1::2] = ...  # TODO: odd dimensions: cosine of position * div_term

        self.register_buffer("pe", pe.unsqueeze(0))   # (1, max_len, d_model)

    def forward(self, x):
        # x: (batch_size, seq_len, d_model)
        return ...  # TODO: add the encodings of the first seq_len positions to x

In [ ]:
# ✅ Check your positional encoding
pos_enc = PositionalEncoding(d_model=8, max_len=50)
assert pos_enc.pe.shape == (1, 50, 8), "pe should have shape (1, max_len, d_model)"
assert len(list(pos_enc.parameters())) == 0, "the encoding must not be trainable"
assert torch.allclose(pos_enc.pe[0, 0], torch.tensor([0.0, 1.0] * 4)), "position 0 should be sin(0) = 0 and cos(0) = 1, alternating"
assert math.isclose(pos_enc.pe[0, 3, 0].item(), math.sin(3), abs_tol=1e-5), "dimension 0 should be sin(pos)"
assert math.isclose(pos_enc.pe[0, 3, 3].item(), math.cos(3 / 10000 ** (2 / 8)), abs_tol=1e-5), "dimension 3 should be the cosine of the second frequency"
x = torch.zeros(2, 5, 8)
assert pos_enc(x).shape == (2, 5, 8) and torch.allclose(pos_enc(x)[1], pos_enc.pe[0, :5]), "forward should add pe[:, :seq_len] to x"
print("Looks good ✅")

**4.4 Run the two cells below.** The first one plots the encoding: one row per position, one column per dimension. The second one repeats your experiment from 4.1, this time with positional encodings added to the embeddings.

In [ ]:
pos_enc = PositionalEncoding(d_model=64, max_len=100)

plt.figure(figsize=(10, 4))
plt.imshow(pos_enc.pe[0], cmap="RdBu", aspect="auto")
plt.xlabel("dimension")
plt.ylabel("position")
plt.colorbar()
plt.title("Sinusoidal positional encoding")
plt.show()

In [ ]:
pos_enc = PositionalEncoding(d_model=8)

X_pos = pos_enc(X.unsqueeze(0))[0]                 # the sentence, with positions
X_shuffled_pos = pos_enc(X[perm].unsqueeze(0))[0]  # the shuffled sentence, with positions

out, _ = scaled_dot_product_attention(X_pos, X_pos, X_pos)
out_shuffled, _ = scaled_dot_product_attention(X_shuffled_pos, X_shuffled_pos, X_shuffled_pos)

print("Same output, in the new order?", torch.allclose(out_shuffled, out[perm], atol=1e-6))

**4.5 Describe the pattern in the heatmap.** How do the left-most and the right-most dimensions differ, and why is it useful to have both kinds?

---

*Your answer here:*

---

## Take-aways

- Attention is a **soft lookup**: compare a query with all keys, turn the scores into weights with a softmax, and average the values. Four lines of code.
- The scaling by $\sqrt{d_k}$ keeps the softmax out of saturation at initialisation, so that gradients can flow.
- Masks set scores to $-\infty$ **before** the softmax. A causal mask hides the future, a padding mask hides `<pad>` keys.
- Self-attention on its own sees a **bag of words**. Word order has to be added to the input with positional encodings.

Continue with `1. Transformer.ipynb`, where these pieces become a full encoder–decoder model.

*Reference: Vaswani et al. (2017), "Attention Is All You Need".*